# **🏡 Project: The House Price Engine 📈**

## **🌟 What are we building?**

Welcome to the **House Price Prediction Project**! Pricing real estate is notoriously tricky. It’s a mix of hard facts (square footage) and soft variables (neighborhood vibes). For banks, buyers, and platforms like Zillow, getting this wrong by even 5% can mean leaving tens of thousands of dollars on the table.

In this project, we are stepping into the shoes of a **Data Scientist** to build a machine learning model that takes the guesswork out of property values. We will use `pandas` to wrangle messy raw housing data and build a predictive engine that accurately prices homes based on their actual features.

---

## **🗺️ The Roadmap: How we get it done**

We aren't just throwing data at an algorithm and hoping for the best. We’re building a clean, step-by-step pipeline across four key phases:

* **1. Digging into the Data (EDA) 🔍**
  * We'll use `pandas` to pull in our CSV files, check out what data types we're dealing with, and hunt down missing values. 
  * We'll look at the distribution of house prices to see if luxury homes are skewing our numbers, and find out which variables actually correlate with a higher price tag.

* **2. Cleaning & Feature Engineering 🛠️**
  * Raw data is never perfect. We will use `pandas` methods to fill in missing gaps and drop weird outliers (like a massive mansion sold for dirt cheap).
  * We'll create smarter features that the model can understand—like combining individual porch and deck metrics into a single "Total Outdoor Space" variable, or calculating exactly how old a house was the year it was sold.

* **3. Training the Models 🤖**
  * Because we are predicting a continuous number (price), this is a **Regression** problem.
  * We’ll start with a straightforward linear model to set a baseline score. Once that’s locked in, we’ll unleash heavy-hitting gradient-boosted trees like **XGBoost** and **LightGBM** to handle the complex, non-linear relationships in the data.

* **4. Keeping Evaluation Realistic 📊**
  * We will test our models using **RMSLE** (Root Mean Squared Log Error). Why? Because a \$20,000 mistake on a \$100,000 starter home is a disaster, but a \$20,000 mistake on a \$2,000,000 mansion is practically a rounding error. Log error keeps our penalties fair across all price brackets.

---

## **💡 Coding Standards**

We are writing code that looks like it belongs in a production environment, not just a sandbox:

* **Readable & Modular:** No giant blocks of messy code. We’ll write clean, reusable python functions with clear descriptions.
* **Bulletproof Integrity:** We will explicitly validate our data shapes and types using `pandas` before passing anything to our machine learning models. 
* **Scalable Thinking:** The logic we write for this dataset will be clean enough to easily scale up to enterprise-level data down the road.

### 🚀 Automated Data Ingestion

To ensure maximum reproducibility and maintain clean versioning, we pull the dataset directly using Kaggle's tools. This automated process fetches the raw housing feature records—tracking structural properties, location metrics, and sales history—directly into our environment.

* **Dataset Credit:** Vedat Gül via Kaggle (*House Prices Prediction / Advanced Regression Techniques*).
* **Source Notebook/Data:** [Kaggle Notebook Link](https://www.kaggle.com/datasets/fratzcan/usa-house-prices)

### 📥 Loading the Dataset and Libraries

Before we start, we need to install the necessary Python libraries and **load the dataset**.


In [112]:
# Install the required libraries
%pip install kagglehub pandas numpy matplotlib seaborn scikit-learn lightgbm xgboost sweetviz scikit-optimize jupyterlab nbconvert imblearn xgboost joblib -q

# Install and update the watermark package to display environment and library version information
%pip install -q -U watermark

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [113]:
# ============================================================
# 📦 DEPENDENCIES
# ============================================================

# ✅ Core
import os
import warnings
import joblib
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore', category=UserWarning)

# ✅ Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import sweetviz as sv
plt.style.use('dark_background')


# ✅ Preprocessing
from sklearn.impute          import SimpleImputer
from sklearn.preprocessing   import StandardScaler, OneHotEncoder
from sklearn.compose         import ColumnTransformer
from sklearn.pipeline        import Pipeline

# ✅ Models
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.linear_model   import Ridge
from sklearn.ensemble        import RandomForestRegressor
import xgboost  as xgb
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score

# ✅ Evaluation
from sklearn.metrics         import root_mean_squared_error, mean_squared_log_error, mean_absolute_error, r2_score

# ✅ Validation
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV


import warnings
import logging

# 1. Ignore Matplotlib font warnings
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")
warnings.filterwarnings("ignore", message=".*findfont.*")

# 2. Quiet down Matplotlib's internal logger
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

In [114]:
# Load the watermark extension to log the environment state
%reload_ext watermark

# Display professional metadata tracking our data engineering stack
%watermark -a "Maykon - 🏡 The House Price Engine" -d -u -v -p pandas,numpy,matplotlib,seaborn,scikit-learn,xgboost,lightgbm

Author: Maykon - 🏡 The House Price Engine

Last updated: 2026-07-21

Python implementation: CPython
Python version       : 3.14.6
IPython version      : 9.15.0

pandas      : 3.0.3
numpy       : 2.5.1
matplotlib  : 3.11.1
seaborn     : 0.13.2
scikit-learn: 1.9.0
xgboost     : 3.3.0
lightgbm    : 4.7.0



In [115]:
# Download latest version
path = kagglehub.dataset_download("fratzcan/usa-house-prices")

print("📦 Path to dataset files:", path)

📦 Path to dataset files: C:\Users\Cibele\.cache\kagglehub\datasets\fratzcan\usa-house-prices\versions\1


In [116]:
# --- LOCATING AND READING THE CSV ---
# List out all files inside the downloaded repository path to spot the target file
all_files = os.listdir(path)
print("📂 Files discovered in directory:", all_files)

# Filter out all CSV files dynamically
csv_files = [file for file in all_files if file.endswith('.csv')]

if len(csv_files) == 0:
    raise FileNotFoundError("❌ Critical Error: No CSV files found in the downloaded folder!")
else:
    # Grab the primary CSV file found
    csv_filename = csv_files[0]
    full_csv_path = os.path.join(path, csv_filename)
    print(f"🎯 Target CSV located: {csv_filename}")

📂 Files discovered in directory: ['USA Housing Dataset.csv']
🎯 Target CSV located: USA Housing Dataset.csv


In [117]:
# Ingest the dataset into a pandas DataFrame
df = pd.read_csv(full_csv_path)
print(f"✅ Dataset successfully loaded! Shape: {df.shape[0]} rows, {df.shape[1]} columns.")

✅ Dataset successfully loaded! Shape: 4140 rows, 18 columns.


In [118]:
# Display the first 5 records to see our column properties and labels
df.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-09 00:00:00,376000.0,3.0,2.00,1340,1384,3.0,0,0,3,1340,0,2008,0,9245-9249 Fremont Ave N,Seattle,WA 98103,USA
1,2014-05-09 00:00:00,800000.0,4.0,3.25,3540,159430,2.0,0,0,3,3540,0,2007,0,33001 NE 24th St,Carnation,WA 98014,USA
2,2014-05-09 00:00:00,2238888.0,5.0,6.50,7270,130017,2.0,0,0,3,6420,850,2010,0,7070 270th Pl SE,Issaquah,WA 98029,USA
3,2014-05-09 00:00:00,324000.0,3.0,2.25,998,904,2.0,0,0,3,798,200,2007,0,820 NW 95th St,Seattle,WA 98117,USA
4,2014-05-10 00:00:00,549900.0,5.0,2.75,3060,7015,1.0,0,0,5,1600,1460,1979,0,10834 31st Ave SW,Seattle,WA 98146,USA


In [119]:
df.tail() #Displays the last 5 rows of the DataFrame df.

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
4135,2014-07-09 00:00:00,308166.666667,3.0,1.75,1510,6360,1.0,0,0,4,1510,0,1954,1979,501 N 143rd St,Seattle,WA 98133,USA
4136,2014-07-09 00:00:00,534333.333333,3.0,2.50,1460,7573,2.0,0,0,3,1460,0,1983,2009,14855 SE 10th Pl,Bellevue,WA 98007,USA
4137,2014-07-09 00:00:00,416904.166667,3.0,2.50,3010,7014,2.0,0,0,3,3010,0,2009,0,759 Ilwaco Pl NE,Renton,WA 98059,USA
4138,2014-07-10 00:00:00,203400.000000,4.0,2.00,2090,6630,1.0,0,0,3,1070,1020,1974,0,5148 S Creston St,Seattle,WA 98178,USA
4139,2014-07-10 00:00:00,220600.000000,3.0,2.50,1490,8102,2.0,0,0,4,1490,0,1990,0,18717 SE 258th St,Covington,WA 98042,USA


In [120]:
df.describe(include='all').T  # Generates descriptive statistics for all numeric columns in your DataFrame.

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
date,4140,68,2014-06-23 00:00:00,142,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,4140.0,NaN,NaN,NaN,553062.877289,583686.452245,0.0,320000.0,460000.0,659125.0,26590000.0
bedrooms,4140.0,NaN,NaN,NaN,3.400483,0.903939,0.0,3.0,3.0,4.0,8.0
bathrooms,4140.0,NaN,NaN,NaN,2.163043,0.784733,0.0,1.75,2.25,2.5,6.75
sqft_living,4140.0,NaN,NaN,NaN,2143.638889,957.481621,370.0,1470.0,1980.0,2620.0,10040.0
sqft_lot,4140.0,NaN,NaN,NaN,14697.638164,35876.838123,638.0,5000.0,7676.0,11000.0,1074218.0
floors,4140.0,NaN,NaN,NaN,1.51413,0.534941,1.0,1.0,1.5,2.0,3.5
waterfront,4140.0,NaN,NaN,NaN,0.007488,0.086219,0.0,0.0,0.0,0.0,1.0
view,4140.0,NaN,NaN,NaN,0.246618,0.790619,0.0,0.0,0.0,0.0,4.0
condition,4140.0,NaN,NaN,NaN,3.452415,0.678533,1.0,3.0,3.0,4.0,5.0


In [121]:
# Information about the dataframe
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4140 entries, 0 to 4139
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           4140 non-null   str    
 1   price          4140 non-null   float64
 2   bedrooms       4140 non-null   float64
 3   bathrooms      4140 non-null   float64
 4   sqft_living    4140 non-null   int64  
 5   sqft_lot       4140 non-null   int64  
 6   floors         4140 non-null   float64
 7   waterfront     4140 non-null   int64  
 8   view           4140 non-null   int64  
 9   condition      4140 non-null   int64  
 10  sqft_above     4140 non-null   int64  
 11  sqft_basement  4140 non-null   int64  
 12  yr_built       4140 non-null   int64  
 13  yr_renovated   4140 non-null   int64  
 14  street         4140 non-null   str    
 15  city           4140 non-null   str    
 16  statezip       4140 non-null   str    
 17  country        4140 non-null   str    
dtypes: float64(4), int6

In [122]:
df.shape # (rows, columns)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 4140
Number of columns: 18


In [123]:
df.dtypes #Displays the data type of each column in the DataFrame.

date                 str
price            float64
bedrooms         float64
bathrooms        float64
sqft_living        int64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
condition          int64
sqft_above         int64
sqft_basement      int64
yr_built           int64
yr_renovated       int64
street               str
city                 str
statezip             str
country              str
dtype: object

In [124]:
df.columns #Returns a list (Index object) containing the names of all columns in the DataFrame.

Index(['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
       'floors', 'waterfront', 'view', 'condition', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'street', 'city',
       'statezip', 'country'],
      dtype='str')

### 🧹 Data Cleaning - processing and handling of missing data.

After loading the dataset and reviewing its structure with 'df.info()' and, the next step is to identify missing values in the dataset.  

We use:

In [125]:
df.isna().sum() # Count missing values per column

date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
street           0
city             0
statezip         0
country          0
dtype: int64

In [126]:
df.drop_duplicates(inplace=True) # Remove duplicate rows from the DataFrame df.
print("Number of duplicate rows Now:", dfc.duplicated().sum()) # After dropping duplicates, check again to confirm that there are no duplicate rows remaining in the DataFrame df.

Number of duplicate rows Now: 0


In [127]:
# this code will standardize the column names by replacing spaces with underscores, converting all characters to lowercase,
# and stripping any leading or trailing whitespace from the column names in the DataFrame df. This is a common practice to ensure that column names 
# are consistent and easier to work with in code.
df.columns = (df.columns.str.replace(' ', '_').str.lower().str.strip())

#### 🧠 Create the profiling report

In [128]:
# 1. Generate the Sweetviz report
report = sv.analyze(df, target_feat='price')

# 2. Save the report to an HTML file
report.show_html('House_Prices_Report.html')

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)

Report House_Prices_Report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


#### 🎯 Defining Features (X) and Target (y)

In supervised machine learning, every dataset is divided into two roles:

- **Features (`X`)** → the input variables the model uses to learn patterns (square footage, location, number of bedrooms).
- **Target (`y`)** → the outcome we want to predict — in our case, **house price**.

> ⚠️ **We split the data here, before outlier removal.** This is a hard rule. Doing any transformation on the full dataset before splitting causes **data leakage** — the model indirectly sees test set information during training, producing results that look great in the notebook but fail in the real world.

In [129]:
# Here we are separating the features (X) from the target variable (y). 
# The target variable is 'price', which indicates the price of the house.
# The features (X) are all the other columns in the DataFrame dfc, which will be used to predict the target variable.
X = df.drop(columns=['price'])  # X = all columns except price
y = df['price']                  # y = ONLY the price column

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### 🧹 Outlier Filtering using Interquartile Range (IQR)

To prevent extreme house prices from distorting model training, we filter out statistical outliers from `y_train` using the **1.5 × IQR rule**:

* **1. Measure Data Spread:**
  * `Q1` (25th percentile) and `Q3` (75th percentile) define the middle 50% range of house prices.
  * `IQR = Q3 - Q1` measures the middle spread of prices.

* **2. Define Acceptable Boundaries:**
  * **Lower Bound:** $Q1 - (1.5 \times IQR)$
  * **Upper Bound:** $Q3 + (1.5 \times IQR)$

* **3. Apply Mask & Filter:**
  * Keep only the houses whose prices fall strictly between the lower and upper thresholds.
  * Apply the resulting boolean mask to **both** `X_train` and `y_train` to maintain matching row indices.

> 🔒 **Data Leakage Safeguard:** Outliers are filtered **strictly on `X_train` / `y_train`** after the train-test split. The test set (`X_test` / `y_test`) is untouched so model evaluation remains unbiased.

In [130]:
# After the split — remove outliers on training data only
before = X_train.shape[0]
Q1 = y_train.quantile(0.25)
Q3 = y_train.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
mask = (y_train >= lower_bound) & (y_train <= upper_bound)
X_train = X_train[mask]
y_train  = y_train[mask]
print(f"Removed {before - X_train.shape[0]} outliers from training set")
print(f"Price range: ${y_train.min():,.0f} — ${y_train.max():,.0f}")

Removed 179 outliers from training set
Price range: $0 — $1,150,000


#### ❓ Missing Value Analysis

To identify incomplete data across the dataset, we calculate both the absolute count and relative percentage of missing (`NaN`) values for each column:

* **1. Measure Missingness:**
  * `df.isna().sum()` counts the total number of missing (`NaN`) entries per feature.
  * `(df.isna().sum() / len(df)) * 100` converts those counts into percentages relative to the total row count.

* **2. Structure & Filter Summary:**
  * Both metrics are combined into a clean pandas `DataFrame` for comparison.
  * `.query('`Missing Count` > 0')` filters out complete columns, displaying only features that actually contain missing data.
  * `.sort_values(by='Missing Count', ascending=False)` orders the results so features with the highest missingness appear first.

> 💡 **Preprocessing Step:** This analysis guides our imputation strategy inside the column transformer pipeline (e.g., deciding whether to impute numeric features using the `median` or categorical features using the `most_frequent` value).

In [131]:
missing_series = df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False)

if not missing_series.empty:
    plt.figure(figsize=(8, 4))
    sns.barplot(x=missing_series.values, y=missing_series.index, palette='Reds_r')
    plt.title('Missing Value Count per Feature')
    plt.xlabel('Count')
    plt.show()
else:
    print("🎉 No missing values found in the dataset!")

🎉 No missing values found in the dataset!


#### ⚙️ Feature Engineering

This is where you create new, more useful columns from what you already have. Based on your dataset's columns, here's what makes sense:

In [132]:
# ============================================================
# 🛠️ FEATURE ENGINEERING
# ============================================================

# Apply feature engineering to both train and test sets separately
# to avoid data leakage — no full dataset transformations after split

for dataset in [X_train, X_test]:
    # Extract sale year from date
    dataset['date']             = pd.to_datetime(dataset['date'])
    dataset['sale_year']        = dataset['date'].dt.year

    # How old was the house when it was sold?
    dataset['house_age']        = dataset['sale_year'] - dataset['yr_built']

    # Was the house ever renovated?
    dataset['was_renovated']    = (dataset['yr_renovated'] > 0).astype(int)

    # How many years since the last update (renovation or original build)?
    dataset['years_since_update'] = dataset.apply(
        lambda row: row['sale_year'] - row['yr_renovated']
        if row['yr_renovated'] > 0
        else row['sale_year'] - row['yr_built'],
        axis=1
    )

# Drop redundant columns — raw values now replaced by engineered features
cols_to_drop = ['country', 'street', 'date', 'yr_built', 'yr_renovated', 'sale_year']
X_train = X_train.drop(columns=cols_to_drop)
X_test  = X_test.drop(columns=cols_to_drop)

# Confirm final shape
print(f"✅ Feature engineering complete!")
print(f"X_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"Final columns : {X_train.columns.tolist()}")

✅ Feature engineering complete!
X_train shape : (3133, 15)
X_test shape  : (828, 15)
Final columns : ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 'city', 'statezip', 'house_age', 'was_renovated', 'years_since_update']


#### 🔍 Preprocessing Feature Alignment Check

Before building the `ColumnTransformer` and training pipelines, we perform a sanity check to verify that all specified feature names exist inside `X_train`:

* **1. List Comprehensions & Set Membership:**
  * `[col for col in numeric_features if col in X_train.columns]` filters `numeric_features` to confirm which columns are present in the training set.
  * `[col for col in numeric_features if col not in X_train.columns]` isolates any numerical features that were accidentally dropped, misspelled, or missing.

* **2. Categorical Column Verification:**
  * The same lookup logic is applied to `categorical_features` to ensure all expected text or group columns exist before sending them to `OneHotEncoder`.

* **3. Diagnostic Visual Feedback:**
  * Clear emoji headers (`✅` and `❌`) make it easy to instantly spot pipeline configuration errors or missing columns in the execution output.

> 💡 **Why This Matters:** Running this check prevents silent pipeline failures, key errors, or unexpected column mismatches during target encoding and scaling steps.

In [133]:
# Separate columns by type
numeric_features = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement',
                    'house_age', 'was_renovated', 'years_since_update']

categorical_features = ['city', 'statezip']

In [134]:
print("✅ Numeric columns check:")
print([col for col in numeric_features if col in X_train.columns])

print("\n❌ Missing numeric columns:")
print([col for col in numeric_features if col not in X_train.columns])

print("\n✅ Categorical columns check:")
print([col for col in categorical_features if col in X_train.columns])

print("\n❌ Missing categorical columns:")
print([col for col in categorical_features if col not in X_train.columns])

✅ Numeric columns check:
['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 'house_age', 'was_renovated', 'years_since_update']

❌ Missing numeric columns:
[]

✅ Categorical columns check:
['city', 'statezip']

❌ Missing categorical columns:
[]


#### ⚡ Pipeline
A pipeline is a sequence of automated steps that processes your data and trains your model in the correct order.
**Instead of manually encoding your data, splitting, and then training — the pipeline chains everything together into a single object. You call fit() once and it handles the rest.**
The biggest advantage is safety — it guarantees that encoding is learned only from training data and applied correctly to test data, eliminating data leakage without you having to think about it.
**In short:**
* Without pipeline → you manage every step manually → easy to make mistakes
* With pipeline    → one object manages everything  → safe and reproducible

In [135]:
# ============================================================
# 🔧 BUILD THE PREPROCESSING PIPELINE
# ============================================================

# --- Numeric pipeline: impute missing values, then scale ---
numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])

# --- Categorical pipeline: impute missing values, then one-hot encode ---
categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])

# Update your ColumnTransformer definition
preprocessor = ColumnTransformer(
    transformers=[('num', numeric_transformer, numeric_features), ('cat', categorical_transformer, categorical_features)], sparse_threshold=0 )

#### 🔁 Cross-Validation and Training the Models

To obtain a more reliable estimate of our model’s performance and reduce the dependency on a single data split, we use cross-validation. This technique repeatedly partitions the dataset into multiple training and validation subsets, allowing the model to be trained and evaluated across different data segments.

* The dataset is divided into $K$ folds (e.g., 5 folds).
* In each iteration:
  * $K - 1$ folds are used for training
  * 1 fold is used for validation

The process is repeated $K$ times, and the final performance is computed as the average metric across all folds.

---

We use **`cross_val_score`** paired with a **`KFold`** cross-validation strategy, setting `shuffle=True` alongside a fixed `random_state` to ensure continuous target values are randomly distributed across all folds for robust evaluation and exact reproducibility.

In [142]:
# Correct setup for Regression tasks (House Prices)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

#### 🏁 Baseline
The baseline represents the minimum threshold your model must surpass.
Before building complex algorithms, we ask: "What score would we get if we made the simplest possible guess without looking at any features?" That is our benchmark.

While a classifier predicts the most frequent category, a DummyRegressor predicts the average target value (e.g., the mean house price) for every single prediction. If our machine learning models cannot beat this simple average benchmark, they aren't learning meaningful patterns from the data.

In [144]:
# Baseline pipeline
dummy_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DummyRegressor(strategy='mean'))
])

# CV evaluation
dummy_mae_scores  = -cross_val_score(dummy_pipeline, X_train, y_train, cv=cv, scoring='neg_mean_absolute_error')
dummy_rmse_scores = -cross_val_score(dummy_pipeline, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error')

print(f"📊 Baseline (DummyRegressor) | MAE: ${dummy_mae_scores.mean():,.0f} (+/- ${dummy_mae_scores.std():,.0f}) | RMSE: ${dummy_rmse_scores.mean():,.0f} (+/- ${dummy_rmse_scores.std():,.0f})")

📊 Baseline (DummyRegressor) | MAE: $178,215 (+/- $5,166) | RMSE: $222,155 (+/- $5,387)


**What these numbers mean**

MAE: $178,215
On average, if your model just predicts the mean price for every house, it's off by $178,215 per house. That's the minimum bar your real model must beat.

RMSE: $222,155
Similar story but RMSE penalizes large errors more heavily. A $222,155 average error means some houses are being missed by much more than $178,215.

+/- $5,166 and +/- $5,387
The standard deviation across the 5 folds is small — meaning the baseline is stable and consistent. The dataset is well distributed across folds.